In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Loading the dataset with House age

df = pd.read_csv('../data/processed/kc_house_data_processed.csv')

#Basic Stats
df.head(5)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,decade,house_age
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,0,1955,0,98178,47.5112,-122.257,1340,5650,1950,70
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,400,1951,1991,98125,47.7210,-122.319,1690,7639,1950,74
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,0,1933,0,98028,47.7379,-122.233,2720,8062,1930,92
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,910,1965,0,98136,47.5208,-122.393,1360,5000,1960,60
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,0,1987,0,98074,47.6168,-122.045,1800,7503,1980,38


In [9]:
# Feature Selection

features = [
    'bedrooms',
    'bathrooms',
    'sqft_living',
    'floors',
    'waterfront',
    'view',
    'condition',
    'grade',
    'house_age',
    'lat',
    'long'
]

print(f"\nFeatures selected: {len(features)}")
print(features)

X = df[features]
y = df['price']

# Missing & Basic Stats
print(f"\nMissing values in X: {X.isnull().sum().sum()}")
print(f"Missing values in y: {y.isnull().sum()}")
print(f"\nDataset size: {len(X)} houses")
print(f"Price range: ${y.min():,.0f} to ${y.max():,.0f}")
print(f"Mean price: ${y.mean():,.0f}")



Features selected: 11
['bedrooms', 'bathrooms', 'sqft_living', 'floors', 'waterfront', 'view', 'condition', 'grade', 'house_age', 'lat', 'long']

Missing values in X: 0
Missing values in y: 0

Dataset size: 21613 houses
Price range: $75,000 to $7,700,000
Mean price: $540,088


In [10]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {len(X_train)} houses & ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing set size: {len(X_test)} houses & ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTraining set price range: ${y_train.min():,.0f} to ${y_train.max():,.0f}")
print(f"Testing set price range: ${y_test.min():,.0f} to ${y_test.max():,.0f}")


Training set size: 17290 houses & (80.0%)
Testing set size: 4323 houses & (20.0%)

Training set price range: $75,000 to $7,700,000
Testing set price range: $82,500 to $5,570,000


In [ ]:
# Feature Scaling

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling completed using StandardScaler.")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

# Example
print("\nExample - sqft_living before/after scaling:")
sqft_idx = features.index('sqft_living')
print(f"  Original (first 5): {X_train.iloc[:5, sqft_idx].values}")
print(f"  Scaled (first 5): {X_train_scaled[:5, sqft_idx]}")



Feature scaling completed using StandardScaler.
Training set shape: (17290, 11)
Testing set shape: (4323, 11)

Example - sqft_living before/after scaling:
  Original (first 5): [1780 1000 1080 2090 1741]
  Scaled (first 5): [-0.32393262 -1.18365301 -1.09547656  0.01775112 -0.36691864]


In [14]:
# Linear Regression Model

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_linear = lr_model.predict(X_train_scaled)
y_test_pred_linear = lr_model.predict(X_test_scaled)

# Model Eval
lr_r2_train = r2_score(y_train, y_train_pred_linear)
lr_r2_test = r2_score(y_test, y_test_pred_linear)
lr_mae_train = mean_absolute_error(y_train, y_train_pred_linear)
lr_mae_test = mean_absolute_error(y_test, y_test_pred_linear)
lr_rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred_linear))
lr_rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_linear))

print("\nLinear Regression Model Training Performance:")
print(f"  R² Train: {lr_r2_train:.4f}")
print(f"  MAE Train: ${lr_mae_train:,.2f}")
print(f"  RMSE Train: ${lr_rmse_train:,.2f}")

print("\nLinear Regression Model Testing Performance:")
print(f"  R² Test: {lr_r2_test:.4f}")
print(f"  MAE Test: ${lr_mae_test:,.2f}")
print(f"  RMSE Test: ${lr_rmse_test:,.2f}")

# Feature Coefficients
feature_imp_linear = pd.DataFrame({
    'Feature': features,
    'Coefficient': lr_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print("\n Top 10 Important Features - Linear Regression:")
print(feature_imp_linear.head(10))



Linear Regression Model Training Performance:
  R² Train: 0.6925
  MAE Train: $125,613.76
  RMSE Train: $200,444.60

Linear Regression Model Testing Performance:
  R² Test: 0.6928
  MAE Test: $128,388.46
  RMSE Test: $215,493.62

 Top 10 Important Features - Linear Regression:
        Feature    Coefficient
2   sqft_living  159734.933843
7         grade  121913.359453
8     house_age   78688.889542
9           lat   75824.511381
4    waterfront   48323.119703
5          view   38464.109092
1     bathrooms   33085.418364
6     condition   15794.419318
3        floors    6975.472528
10         long   -8789.776628


In [15]:
# Ridge Regression Model

alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]
best_alpha = None
best_r2 = -np.inf

# Different alphas values

for alpha in alphas:
    ridge_temp = Ridge(alpha=alpha)
    ridge_temp.fit(X_train_scaled, y_train)
    r2_temp = r2_score(y_test, ridge_temp.predict(X_test_scaled))
    print(f"Ridge Regression with alpha={alpha}: R² Test = {r2_temp:.4f}")
    if r2_temp > best_r2:
        best_r2 = r2_temp
        best_alpha = alpha
print(f"\nBest alpha for Ridge Regression: {best_alpha} with R² Test = {best_r2:.4f}")

ridge_model = Ridge(alpha=best_alpha)
ridge_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_ridge = ridge_model.predict(X_train_scaled)
y_test_pred_ridge = ridge_model.predict(X_test_scaled)

# Model Eval
ridge_r2_train = r2_score(y_train, y_train_pred_ridge)
ridge_r2_test = r2_score(y_test, y_test_pred_ridge)
ridge_mae_train = mean_absolute_error(y_train, y_train_pred_ridge)
ridge_mae_test = mean_absolute_error(y_test, y_test_pred_ridge)
ridge_rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
ridge_rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))

print("\nRidge Regression Model Training Performance:")
print(f"  R² Train: {ridge_r2_train:.4f}")
print(f"  MAE Train: ${ridge_mae_train:,.2f}")
print(f"  RMSE Train: ${ridge_rmse_train:,.2f}")

print("\nRidge Regression Model Testing Performance:")
print(f"  R² Test: {ridge_r2_test:.4f}")
print(f"  MAE Test: ${ridge_mae_test:,.2f}")
print(f"  RMSE Test: ${ridge_rmse_test:,.2f}")

# Feature Coefficients
feature_imp_ridge = pd.DataFrame({
    'Feature': features,
    'Coefficient': ridge_model.coef_
}).sort_values(by='Coefficient', ascending=False)
print("\n Top 10 Important Features - Ridge Regression:")
print(feature_imp_ridge.head(10))

Ridge Regression with alpha=0.1: R² Test = 0.6928
Ridge Regression with alpha=1.0: R² Test = 0.6928
Ridge Regression with alpha=10.0: R² Test = 0.6928
Ridge Regression with alpha=100.0: R² Test = 0.6926
Ridge Regression with alpha=1000.0: R² Test = 0.6899

Best alpha for Ridge Regression: 0.1 with R² Test = 0.6928

Ridge Regression Model Training Performance:
  R² Train: 0.6925
  MAE Train: $125,613.49
  RMSE Train: $200,444.60

Ridge Regression Model Testing Performance:
  R² Test: 0.6928
  MAE Test: $128,388.21
  RMSE Test: $215,493.68

 Top 10 Important Features - Ridge Regression:
        Feature    Coefficient
2   sqft_living  159732.906973
7         grade  121913.066645
8     house_age   78688.082528
9           lat   75824.333563
4    waterfront   48322.907885
5          view   38464.472768
1     bathrooms   33086.005977
6     condition   15794.480720
3        floors    6975.422209
10         long   -8789.753193
